# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Two questions I would ask about the FlyRank research paper, then the same lens turned on my own
Week-5 model: what an honest split costs, what my features would leak if I let them, and which of
my own sentences the evidence does not carry.

The paper is `docs/flyrank-seo-research-march-2026.pdf`. It is not mine to grade, and it discloses
more about its own limits than most public reports do. The point of section 1 is to practise reading
it the way I want my own work read.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #1, "The Anatomy of Growing Content" (CONFIRMED)

The claim: growing pages are 37.6% longer than declining ones (3.2K vs 2.3K words) and 20% younger
(184 vs 230 days), across 74,187 rising and 45,272 falling pages.

**Where the label comes from.** The paper says it plainly on its "How to read" page: trend direction
is computed from the 30-day-vs-previous-30-day impression change, up above +10%, down below -10%,
with stable, flat and new as separate buckets. So the two cohorts being compared are the two tails of
one continuous distribution with the middle removed. That is a reasonable design choice, and it does
mean any gap between the tails reads larger than the same gap measured across the full range.

**My question: is the comparison pooled across brands, and would it survive grouping?** The cohorts
pool 57 brands. If brands differ in both house style and momentum, say one large brand publishes long
pages and happens to be growing, a portfolio-level gap can be a brand-mix artifact rather than a
property of pages. The paper's own data guidance points the same way: prefer per-client windows over
one global cut. The code below runs that check on the 32-client starter slice I have. It is a
different and much smaller sample, so it cannot confirm or refute the paper's number. What it can show
is whether the question is worth asking, and on my slice the pooled and within-client answers point in
opposite directions.

**A second question: when is word count measured?** Word count is read at snapshot time, after the
trend window. The paper's own recommended action is to expand thin pages, so a page that was expanded
during the window would be counted as long and growing. Length would then be partly an outcome of the
same period rather than a pre-existing trait. A refresh-date cut, or word count as of the window
start, would separate the two readings.

### ML Appendix, "What Predicts Growth?"

The claim: a logistic regression separates growing from declining pages with 71% holdout accuracy,
with content age the strongest negative signal and days-visible and recent impressions among the
strongest positive ones. The methods note records an 80/20 split on 61,790 active-content records.

The paper already fences this in: the ML pages are labelled exploratory appendix material that does
not override direct portfolio evidence, and the sibling Random Forest page volunteers that health
score is partly built from its own inputs, so importance is descriptive rather than causal. That is
the disclosure I would want. My questions are about what a reader needs in order to size the number.

**Where the label comes from.** The same trend direction as Finding #1, so the same impression-change
definition, on an active-content subset with impressions and sessions above zero.

**My question: 71% against what floor?** Accuracy only means something next to its base rate. From the
paper's own Finding #1 counts, 74,187 rising against 45,272 falling, always answering "rising" scores
about 62%. If the appendix sample has a similar balance, then 71% is roughly nine points of skill, not
seventy-one. The appendix sample is a different cut, so I cannot compute its floor from the outside,
which is exactly why printing it beside the accuracy would settle the question in one line.

**My question: does an 80/20 split answer the deployment question?** The methods note says 80/20 and
does not mention grouping by brand. With 57 brands and many pages each, a random 80/20 puts pages from
the same brand on both sides, so a model can score well by recognising the brand rather than by
reading the page. Section 2 measures exactly that gap on my own model, and it is worth 0.06 AUC.

**My question: do the features sit before the label window?** Recent impressions and days-visible are
measured over windows that overlap the 30-day window the label is computed from. Section 3 shows what
that costs on my data: adding last-30-day counts next to my legal prior-30-day counts takes AUC from
0.63 to 0.95, because the pair reconstructs the ratio the label is thresholded from.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

ud = df[df["trend_direction"].str.lower().isin(["up", "down"])].copy()
ud["growing"] = ud["trend_direction"].str.lower().eq("up")
print("starter slice: rising", int(ud["growing"].sum()), " falling", int((~ud["growing"]).sum()),
      " clients", ud["client_id"].nunique())
print("always-answer-the-majority accuracy on this up/down cut:",
      round(max(ud["growing"].mean(), 1 - ud["growing"].mean()), 3))
print("paper's own cohort counts imply a floor of:", round(74187 / (74187 + 45272), 3))

rows = []
for col in ["word_count", "content_age_days"]:
    pooled = ud[ud["growing"]][col].mean() - ud[~ud["growing"]][col].mean()
    per = ud.groupby(["client_id", "growing"])[col].mean().unstack().dropna()
    within = per[True] - per[False]
    rows.append([col, round(pooled, 1), round(within.median(), 1), len(within),
                 int((np.sign(within) == np.sign(pooled)).sum())])
print()
print(pd.DataFrame(rows, columns=["column", "pooled gap (growing - declining)",
                                  "median within-client gap", "clients with both",
                                  "clients agreeing with pooled sign"]).to_string(index=False))

starter slice: rising 4388  falling 16262  clients 30
always-answer-the-majority accuracy on this up/down cut: 0.788
paper's own cohort counts imply a floor of: 0.621

          column  pooled gap (growing - declining)  median within-client gap  clients with both  clients agreeing with pooled sign
      word_count                            -223.8                      26.2                 30                                 10
content_age_days                              52.3                       0.7                 30                                 16


On my 32-client slice the pooled comparison says growing pages are about 224 words **shorter** and 52
days **older** than declining ones, which is the opposite of the paper's direction on its much larger
portfolio. Two honest reasons that is not a refutation: this slice is 30K rows against the paper's
341K, and its cohort balance is inverted (4,388 rising against 16,262 falling, where the paper had far
more rising than falling). Different sample, different answer is expected.

The part that travels is the grouping behaviour. Pooled, the word-count gap says growing pages are
shorter. Measured inside each client and then compared, it flips: growing pages are longer in 20 of the
30 clients that carry both cohorts, median +26 words, so only 10 clients agree with the pooled sign.
The age gap does not flip, it evaporates: 52 days pooled becomes a median of 0.7 days within clients.
Either way the aggregate is being moved by which clients contribute pages to each cohort rather than by
what the pages are like, which is the signature of a mix effect. That is the concrete reason I would
want a cohort comparison like this repeated inside brands before the length gap is read as a property
of pages, and it is the same reason my own model is scored with whole clients held out in section 2.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model is a Logistic Regression over leakage-safe features, ranking visible pages by decline
risk. Here it is scored twice on the same data with the same features, changing only the split.

**Before**: random 5-fold, which ignores `client_id` and lets pages from one client sit on both sides
of the split. **After**: 5-fold with whole clients held out, so every test page belongs to a client the
model has never seen. A time-aware split is not available to me here: the starter table is one snapshot
per page rather than a history, so there is no row-level time axis to cut on. The time ordering lives
inside the columns instead, which is why the prior-30-day window is the only traffic window I allow as
a feature.

The base rate sits next to every number, because 0.598 of visible pages are declining and any accuracy
has to be read against that.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import KFold

num = ["content_age_days", "days_since_last_update", "word_count", "char_count",
       "search_volume", "competition", "cpc",
       "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
cats = ["content_type", "main_intent", "competition_level"]

def features(frame, extra=[]):
    f = frame.copy()
    for c in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
        f["log_" + c] = np.log1p(f[c].fillna(0))
    f["has_keyword"] = f["search_volume"].notna().astype(float)
    f["has_word_count"] = f["word_count"].notna().astype(float)
    ncols = num + ["log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d",
                   "has_keyword", "has_word_count"] + extra
    return pd.concat([f[ncols], f[cats].astype("object").fillna("unknown")], axis=1), ncols

def model(ncols):
    pre = ColumnTransformer([
        ("n", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value=0)),
                        ("sc", StandardScaler())]), ncols),
        ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats)])
    return Pipeline([("pre", pre), ("lr", LogisticRegression(class_weight="balanced",
                     max_iter=2000, C=1.0, random_state=42))])

def precision_at_k(y, scores, k=50):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(y)[order[:k]].mean())

def grouped_folds(groups, seed, n=5):
    gs = pd.Series(groups)
    order = np.random.default_rng(seed).permutation(np.array(sorted(gs.unique())))
    sizes = gs.value_counts()
    load = np.zeros(n)
    fold_of = {}
    for c in order:
        fi = int(np.argmin(load))
        fold_of[c] = fi
        load[fi] += sizes[c]
    fid = gs.map(fold_of).values
    return [(np.where(fid != f)[0], np.where(fid == f)[0]) for f in range(n)]

vis = df[df["impressions_90d"] >= 100].reset_index(drop=True)
y_all = vis["is_declining"].values

def run(folds, extra=[]):
    a, p, acc = [], [], []
    for tr, te in folds:
        Xtr, ncols = features(vis.iloc[tr], extra)
        Xte, _ = features(vis.iloc[te], extra)
        m = model(ncols)
        m.fit(Xtr, y_all[tr])
        s = m.predict_proba(Xte)[:, 1]
        a.append(roc_auc_score(y_all[te], s))
        p.append(precision_at_k(y_all[te], s))
        acc.append(accuracy_score(y_all[te], (s > 0.5).astype(int)))
    return np.mean(a), np.mean(p), np.mean(acc)

random_folds = list(KFold(5, shuffle=True, random_state=42).split(vis))
client_folds = grouped_folds(vis["client_id"].values, 0)
before, after = run(random_folds), run(client_folds)

print("base rate (always answer 'declining'):", round(y_all.mean(), 3))
print(pd.DataFrame([
    ["before: random 5-fold (client ignored)", *np.round(before, 3)],
    ["after: client-grouped 5-fold", *np.round(after, 3)],
    ["gap (memorisation removed)", *np.round(np.array(after) - np.array(before), 3)],
], columns=["split", "ROC_AUC", "P@50", "accuracy"]).to_string(index=False))

base rate (always answer 'declining'): 0.598
                                 split  ROC_AUC   P@50  accuracy
before: random 5-fold (client ignored)    0.685  0.904     0.647
          after: client-grouped 5-fold    0.625  0.800     0.627
            gap (memorisation removed)   -0.060 -0.104    -0.021


The random split reads 0.685 AUC and 0.904 P@50. The honest split reads 0.625 and 0.800. The gap, 0.06
of AUC and 0.10 of top-50 precision, is not a model improvement or a model failure. It is the amount of
my earlier number that was client memorisation: pages from one client share templates, a niche and a
tracking setup, so a model that has seen half a client's pages can recognise the rest without learning
anything about decline.

The accuracy column makes the same point in the other direction. 0.647 looks like a passing grade until
it sits next to the 0.598 base rate, at which point it is under five points of skill. This is the number
I would want printed beside any headline accuracy, including the appendix's 71%.

Everything I report from here uses the grouped split. The honest number is the smaller one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The label is built from the 30-day-vs-previous-30-day impression change, so the hunt is for any feature
that can see into that window. Working the taxonomy against my final feature set:

**1. Label-derived features.** `trend_direction` and `trend_pct` are the label and its magnitude. Both
are excluded. The test below adds `trend_pct` back and AUC goes to 0.998, which is the confession the
taxonomy predicts.

**2. Future or overlapping windows.** This is the real risk in my table, and it is subtler than one bad
column. My legal features include `impressions_prev_30d`, the window *before* the label window. If I
also admit `impressions_last_30d`, neither column is the label on its own, but the pair reconstructs the
ratio the label is thresholded from. The check below shows AUC jumping from 0.625 to 0.946 on that pair
alone. Every 90-day column is excluded for the same reason: it contains the label's window.

**3. Decision-derived features.** `provider_used` and `model_used` record that someone already ran this
page through a generation or refresh workflow. That is a decision, not a property of the world, and
using it would teach the model the old routing rule. They are excluded. I did test them during Week 5
and they made the model worse on held-out clients, which is a convenient confirmation but not the
reason. `client_id` is a pseudonym and is used only to group the split, never as a feature.

**Population selection.** This is the one I have to disclose rather than defend. My population is
`impressions_90d >= 100`, and `impressions_90d` is a label-window quantity. So the set of rows I score
is chosen using information from the outcome window. It is defensible, ranking pages nobody sees is not
a refresh decision, and the visible cut is the population the queue actually serves. But it is a choice
made with outcome-window information, and both the paper and I should say so out loud rather than let it
pass as neutral. `impressions_90d` never enters the model as a feature.

**Missingness.** The data notes warn that missingness tracks `content_type`, so a blind `fillna(0)`
smuggles in a category signal. I fill with zero *and* carry `has_keyword` / `has_word_count` flags so
the model can tell "no data" from "a real zero".

In [3]:
suspects = [("honest features only (my final set)", []),
            ("+ ctr, avg_position", ["ctr", "avg_position"]),
            ("+ 90d traffic", ["impressions_90d", "clicks_90d"]),
            ("+ last-30d counts (pairs with prev-30d)", ["impressions_last_30d", "clicks_last_30d",
                                                         "sessions_last_30d"]),
            ("+ trend_pct (the label's own magnitude)", ["trend_pct"])]
rows = []
for label, extra in suspects:
    auc, p50, acc = run(client_folds, extra)
    rows.append([label, round(auc, 3), round(p50, 3)])
print("train WITH vs WITHOUT each suspect, client-grouped folds, base rate",
      round(y_all.mean(), 3))
print(pd.DataFrame(rows, columns=["feature set", "ROC_AUC", "P@50"]).to_string(index=False))

checks = [
    ("Timeline drawn, features strictly before the label window", "yes, prior-30d only"),
    ("No label-derived or sibling columns", "yes, trend_* excluded"),
    ("No product flags / existing-system scores", "yes, provider_used and model_used excluded"),
    ("Population selection checked for outcome-window info", "NO, impressions_90d gate: disclosed"),
    ("Split grouped by the repeating entity", "yes, whole clients"),
    ("Base rate printed next to every metric", "yes, 0.598"),
    ("Top importance sanity-checked, 'too good' investigated", "yes, see table above"),
    ("Metrics out-of-fold, never in-sample", "yes"),
    ("Sealed holdout leaves receipts", "yes, work/experiments/lockbox.json + ledger.jsonl"),
]
print()
print(pd.DataFrame(checks, columns=["attack checklist", "verdict"]).to_string(index=False))

train WITH vs WITHOUT each suspect, client-grouped folds, base rate 0.598
                            feature set  ROC_AUC  P@50
    honest features only (my final set)    0.625 0.800
                    + ctr, avg_position    0.641 0.840
                          + 90d traffic    0.664 0.916
+ last-30d counts (pairs with prev-30d)    0.946 1.000
+ trend_pct (the label's own magnitude)    0.998 1.000

                                         attack checklist                                           verdict
Timeline drawn, features strictly before the label window                               yes, prior-30d only
                      No label-derived or sibling columns                             yes, trend_* excluded
                No product flags / existing-system scores        yes, provider_used and model_used excluded
     Population selection checked for outcome-window info               NO, impressions_90d gate: disclosed
                    Split grouped by the repeating enti

The escalation is the whole audit in one table. Ratios computed over the label's own window buy
almost nothing on their own (`ctr` and `avg_position` move AUC from 0.625 to 0.641, because knowing a
page's click rate says little about which way its impressions moved). The damage comes from the pair:
`impressions_last_30d` next to my legal `impressions_prev_30d` is the trend ratio in two columns, and
AUC jumps to 0.946. `trend_pct` finishes the job at 0.998.

The lesson I take from this is that a leakage audit cannot be a column-by-column blacklist. Each of
those columns is individually harmless-looking, and `impressions_prev_30d` is genuinely legal. What
makes them dangerous is the arithmetic between them and the label definition. That is why my rule is
stated as a window rule rather than a column list: only the sub-window strictly before the label window
is admissible.

The one item my checklist does not clear is the population gate, and the honest resolution is to
disclose it in the limitations rather than to quietly keep it.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentence, from `w05_model.ipynb` section 5:

> "On clients the search never touched, widened training holds its AUC and moves Precision@50 up by a
> clear margin. That is the result I would stand behind: it generalises to new clients on the metric the
> refresh queue is built from."

Two things in there go further than my evidence. "By a clear margin" describes a single point estimate,
0.76 to 0.84, computed on one draw of six held-out clients, and top-50 precision is measured on fifty
rows. "It generalises to new clients" is a general claim built on one sample of clients. So I put an
interval on it: a cluster bootstrap that resamples the six lockbox clients, since clients are the unit
that repeats, not pages.

The interval is below. The difference is +0.08 with a 95% interval of roughly [-0.02, +0.12], which
includes zero. 78% of resamples favour the widened model, so the direction is consistent, but at six
clients I cannot separate the effect from sampling noise. My sentence claimed more than that.

**Rewritten:**

> On six held-out clients, top-50 precision was measured at 0.84 for the widened model against 0.76 for
> the visible-train model, a difference of +0.08. A cluster bootstrap over those six clients puts the
> 95% interval on that difference at [-0.02, +0.12], so the direction is consistent (78% of resamples
> favour widening) but the effect is not separable from sampling noise at this number of clients. Read
> it as directional, decision-support evidence for widening the training population, not as a measured
> improvement. A firmer read needs more held-out clients, not more tuning.

The same edit applies to how I talk about the model overall. "The model finds declining pages" becomes
"on held-out clients the model ranked declining pages above the base rate of 0.598, with observed AUC
0.625 and roughly four in five of its top fifty truly declining". Decision-support for an editor
triaging a refresh queue, not a prediction of why any page declines.

In [4]:
import json

lockbox = set(json.load(open("../experiments/lockbox.json"))["clients"])
dev_all = df[~df["client_id"].isin(lockbox)]
dev_vis = dev_all[dev_all["impressions_90d"] >= 100]
lb = df[(df["client_id"].isin(lockbox)) & (df["impressions_90d"] >= 100)].reset_index(drop=True)
Xlb, _ = features(lb)
y_lb = lb["is_declining"].values

scores = {}
for name, tr in [("visible-train", dev_vis), ("widened-train", dev_all)]:
    Xtr, ncols = features(tr)
    m = model(ncols)
    m.fit(Xtr, tr["is_declining"].values)
    scores[name] = m.predict_proba(Xlb)[:, 1]

point = {k: precision_at_k(y_lb, s) for k, s in scores.items()}
print("lockbox clients:", len(lockbox), " visible pages:", len(lb),
      " base rate:", round(y_lb.mean(), 3))
print("point estimate P@50: visible-train", round(point["visible-train"], 3),
      " widened-train", round(point["widened-train"], 3),
      " difference", round(point["widened-train"] - point["visible-train"], 3))

cl = lb["client_id"].values
uc = np.array(sorted(set(cl)))
rng = np.random.default_rng(7)
draws = {"visible-train": [], "widened-train": [], "difference": []}
for _ in range(2000):
    pick = rng.choice(uc, len(uc), replace=True)
    idx = np.concatenate([np.where(cl == c)[0] for c in pick])
    yb = y_lb[idx]
    if len(np.unique(yb)) < 2:
        continue
    a = precision_at_k(yb, scores["visible-train"][idx])
    b = precision_at_k(yb, scores["widened-train"][idx])
    draws["visible-train"].append(a)
    draws["widened-train"].append(b)
    draws["difference"].append(b - a)

print("\ncluster bootstrap over the 6 lockbox clients, B =", len(draws["difference"]))
print(pd.DataFrame([[k, round(np.percentile(v, 2.5), 3), round(np.percentile(v, 97.5), 3)]
                    for k, v in draws.items()],
                   columns=["quantity", "95% low", "95% high"]).to_string(index=False))
print("share of resamples favouring widened-train:",
      round(float(np.mean(np.array(draws["difference"]) > 0)), 2))

lockbox clients: 6  visible pages: 5284  base rate: 0.605
point estimate P@50: visible-train 0.76  widened-train 0.84  difference 0.08



cluster bootstrap over the 6 lockbox clients, B = 2000
     quantity  95% low  95% high
visible-train     0.64      0.88
widened-train     0.64      0.94
   difference    -0.02      0.12
share of resamples favouring widened-train: 0.78


The interval does the arguing. Both models' top-50 precision sits somewhere between roughly 0.64 and
0.94 depending on which six clients you draw, and the difference between them cannot be pinned away
from zero with six clients. That does not make widening a bad idea: it won on the development folds, it
won on fresh folds, and it wins 78% of the resamples here. It makes the honest version of the sentence a
directional one, and it tells me the next useful thing to buy is more held-out clients rather than more
model tuning.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card